In [11]:
import csv
import json
import os
import shutil
import re

from pathlib import Path
from collections import Counter

# Student Information
ENGINEER = "Olermo"
STUDENT_ID = "TUPM-26-1925"
SEED_DIGIT = 5
ID_SUM = 25

# Student-specific rules
RECORD_LIMIT = 1000 + ID_SUM
TARGET_STATUS_CLASS = (SEED_DIGIT % 4) + 2

# Corrected file paths
PROJECT_NAME = "Lab6_File_Handling_Additional_Exercises"

cwd = Path.cwd()

if cwd.name == PROJECT_NAME:
    PROJECT_DIR = cwd
elif cwd.name.startswith("Exercise") and cwd.parent.name == PROJECT_NAME:
    PROJECT_DIR = cwd.parent
elif (cwd / PROJECT_NAME / "data").is_dir():
    PROJECT_DIR = cwd / PROJECT_NAME
else:
    PROJECT_DIR = next(
        (p for p in cwd.parents if p.name == PROJECT_NAME),
        None
    )

if PROJECT_DIR is None:
    raise FileNotFoundError("Project folder not found.")

EXERCISE_DIR = PROJECT_DIR / "Exercise1_Web_Server_Logs"

DATA_FILE = PROJECT_DIR / "data" / "weblog.csv"

OUTPUT_DIR = EXERCISE_DIR / "output"
BACKUP_DIR = EXERCISE_DIR / "backup"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATA_FILE)
print("Dataset exists:", DATA_FILE.exists())

Dataset: d:\Downloads\OLERMO\Lab6_File_Handling_Olermo\Lab6_File_Handling_Additional_Exercises\data\weblog.csv
Dataset exists: True


In [3]:
def normalize_column(name):
    return re.sub(r"[^a-z0-9]", "", name.lower())


def find_column(headers, possible_names):
    normalized = {
        normalize_column(header): header
        for header in headers
    }

    for name in possible_names:
        key = normalize_column(name)

        if key in normalized:
            return normalized[key]

    raise ValueError(
        f"Required column not found. Expected one of: {possible_names}"
    )

In [4]:
filtered_records = []

status_counts = Counter()
url_counts = Counter()

records_processed = 0
invalid_records = 0

try:
    with open(DATA_FILE, "r", encoding="utf-8-sig", newline="") as file:

        reader = csv.DictReader(file)

        headers = reader.fieldnames

        ip_col = find_column(headers, ["IP"])
        time_col = find_column(headers, ["Time"])
        url_col = find_column(headers, ["URL"])
        status_col = find_column(
            headers, ["Status", "Staus", "HTTP Status"]
        )

        for row in reader:

            if records_processed >= RECORD_LIMIT:
                break

            records_processed += 1

            ip = row.get(ip_col)
            timestamp = row.get(time_col)
            url = row.get(url_col)
            status = row.get(status_col)

            if not all([ip, timestamp, url, status]):
                invalid_records += 1
                continue

            match = re.match(r"\s*(\d{3})", str(status))

            if not match:
                invalid_records += 1
                continue

            status_code = match.group(1)

            if int(status_code[0]) == TARGET_STATUS_CLASS:

                filtered_records.append(row)

                status_counts[status_code] += 1
                url_counts[url] += 1

except FileNotFoundError:
    print("Dataset file not found:", DATA_FILE)

except Exception as error:
    print("Error:", error)


print("Records processed:", records_processed)
print("Invalid records:", invalid_records)
print("Records in target class:", len(filtered_records))
print("Status code counts:", dict(status_counts))

Dataset file not found: ..\data\web_log.csv
Records processed: 0
Invalid records: 0
Records in target class: 0
Status code counts: {}


In [5]:
top_five_urls = url_counts.most_common(5)

print("Five most requested URLs:")

for url, count in top_five_urls:
    print(url, ":", count)

Five most requested URLs:


In [6]:
filtered_csv = OUTPUT_DIR / "filtered_web_logs.csv"

if filtered_records:

    with open(filtered_csv, "w", encoding="utf-8", newline="") as file:

        writer = csv.DictWriter(
            file,
            fieldnames=filtered_records[0].keys()
        )

        writer.writeheader()
        writer.writerows(filtered_records)

    print("Filtered CSV successfully created.")

else:
    print("No records found in the target status class.")

No records found in the target status class.


In [7]:
summary = {
    "student": ENGINEER,
    "student_id": STUDENT_ID,
    "records_processed": records_processed,
    "target_status_class": f"{TARGET_STATUS_CLASS}xx",
    "records_in_target_class": len(filtered_records),
    "count_per_status_code": dict(status_counts),
    "five_most_requested_urls": [
        {"url": url, "count": count}
        for url, count in top_five_urls
    ],
    "invalid_records": invalid_records
}

json_file = OUTPUT_DIR / "web_log_summary.json"

with open(json_file, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("JSON summary created.")

JSON summary created.


In [8]:
backup_file = BACKUP_DIR / "web_log_summary_backup.json"

shutil.copy2(json_file, backup_file)

if backup_file.exists():

    with open(json_file, "rb") as original:
        original_content = original.read()

    with open(backup_file, "rb") as backup:
        backup_content = backup.read()

    if original_content == backup_content:
        print("Backup verified successfully.")
    else:
        print("Backup verification failed.")

else:
    print("Backup file does not exist.")

Backup verified successfully.


In [9]:
print("\n===== WEB SERVER ACCESS LOG ANALYSIS =====")

print("Records processed:", records_processed)
print("Target status class:", f"{TARGET_STATUS_CLASS}xx")
print("Records in target class:", len(filtered_records))
print("Count per status code:", dict(status_counts))
print("Five most requested URLs:", top_five_urls)
print("Filtered CSV:", filtered_csv)
print("JSON summary:", json_file)
print("Backup verified:", backup_file.exists())


===== WEB SERVER ACCESS LOG ANALYSIS =====
Records processed: 0
Target status class: 3xx
Records in target class: 0
Count per status code: {}
Five most requested URLs: []
Filtered CSV: output\filtered_web_logs.csv
JSON summary: output\web_log_summary.json
Backup verified: True
